In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Mini CCNet-style pipeline, reusing cc_net functions where possible.

Features:
1. Read documents from final_uniform_replace.jsonl (one JSON per line = one document)
2. Perform document-level deduplication using cc_net.normalize_for_dedup
3. Perform LID using cc_net.split_by_lang.Classifier
4. Compute perplexity + bucket with cc_net.perplexity.MultiSentencePiece + DocLM + PerplexityBucket
5. Compute retention rate of zero-width characters / watermark characters
"""

# ============================================================
# 0️⃣ Dependency installation (Use pip in scripts; in Colab use !pip install)
# ============================================================
# Upgrade pip (optional)
!pip install --upgrade pip >/dev/null

# 1. Install kenlm (from GitHub; default branch is master)
!pip install git+https://github.com/kpu/kenlm.git >/dev/null

# 2. Install other dependencies
!pip install fasttext sentencepiece tqdm >/dev/null

# 3. Clone cc_net directly (no need to specify @master)
!git clone https://github.com/facebookresearch/cc_net.git >/dev/null

# 4. Add cc_net to sys.path
import sys, os
sys.path.append("/content/cc_net")  # Default clone path in Colab

# Verify import
from cc_net import text_normalizer, split_by_lang, perplexity
print("✅ cc_net imported OK")


import os
import json
import gzip
import re
import hashlib
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
from tqdm import tqdm

# cc_net original functions
from cc_net import text_normalizer
from cc_net import split_by_lang, perplexity


# ============================================================
# 1️⃣ Path configuration
# ============================================================
INPUT_FILE = Path("/content/final_uniform_replace.jsonl")
OUTPUT_DIR = Path("/content/cleaned_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_FILE = OUTPUT_DIR / "cleaned_ccnet.jsonl.gz"
DROPPED_FILE = OUTPUT_DIR / "dropped_ccnet.jsonl"
PPL_FILE = OUTPUT_DIR / "cleaned_ccnet_with_ppl.jsonl.gz"

LID_MODEL_PATH = Path("/content/lid.176.bin")  # fastText LID model
SP_MODEL_DIR = Path("/content/ccnet_models")
SP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SP_MODEL = SP_MODEL_DIR / "en.sp.model"
LM_MODEL = SP_MODEL_DIR / "en.arpa.bin"
CUTOFF_CSV = SP_MODEL_DIR / "cutoff.csv"

MYTEXT_PATH = Path("/content/myText.txt")

print(f"✅ Using input: {INPUT_FILE}")
print(f"✅ Output: {CLEANED_FILE}")


# ============================================================
# 2️⃣ fastText + NumPy 2 compatibility patch
#    (Some fastText versions use np.assarray / np.array(copy=False))
# ============================================================
if not hasattr(np, "assarray"):
    np.assarray = np.asarray  # compatibility patch

_old_np_array = np.array  # backup


def _safe_array(obj, *args, **kwargs):
    # Remove "copy=False" to avoid NumPy 2.x incompatibility
    if "copy" in kwargs and kwargs["copy"] is False:
        kwargs.pop("copy")
    return _old_np_array(obj, *args, **kwargs)


np.array = _safe_array  # patched version


# ============================================================
# 3️⃣ Zero-width character counting utility
# ============================================================
ZWC_RE = re.compile(r'[\u200b\u200c\u200d\ufeff]')


def count_zwc(text: str) -> int:
    return len(ZWC_RE.findall(text or ""))


# ============================================================
# 4️⃣ Load documents (one JSON per line)
#    Also assign stable wm_idx for watermarked docs (1-to-1 mapping)
# ============================================================
def load_docs(input_file: Path) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Returns:
    docs: cleaned document list (text, is_watermarked, wm_idx)
    raw_wm_docs: all original is_watermarked=True docs (for raw stats)
    """
    docs = []
    raw_wm_docs = []

    wm_idx_counter = 0

    with input_file.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                raw = json.loads(line)
            except Exception:
                continue

            text = raw.get("watermarked", raw.get("text", ""))
            is_wm = bool(raw.get("is_watermarked", False))

            wm_idx = None
            if is_wm:
                wm_idx = wm_idx_counter
                wm_idx_counter += 1
                raw_wm_docs.append(
                    {
                        "wm_idx": wm_idx,
                        "watermarked": raw.get("watermarked", text or ""),
                    }
                )

            docs.append(
                {
                    "text": text or "",
                    "is_watermarked": is_wm,
                    "wm_idx": wm_idx,
                }
            )

    print(f"📥 Loaded {len(docs)} docs, {len(raw_wm_docs)} raw watermarked docs.")
    return docs, raw_wm_docs


# ============================================================
# 5️⃣ Document-level dedup (using cc_net.normalize_for_dedup)
# ============================================================
def normalize_for_dedup(text: str) -> str:
    # Wrapper for cc_net.text_normalizer.normalize_for_dedup
    return text_normalizer.normalize_for_dedup(text or "")


def ccnet_doc_dedup(
    docs: List[Dict[str, Any]], field: str = "text"
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Document-level dedup:
    Entire text is normalized then hashed (SHA1).
    Uses cc_net.normalize_for_dedup.
    """
    seen_hashes = set()
    kept, dropped = [], []

    for doc in tqdm(docs, desc="🔹 CCNet-style doc dedup"):
        raw = doc.get(field, "")
        if not raw.strip():
            doc["is_duplicate"] = True
            dropped.append(doc)
            continue

        norm = normalize_for_dedup(raw)
        if not norm:
            doc["is_duplicate"] = True
            dropped.append(doc)
            continue

        h = hashlib.sha1(norm.encode("utf-8")).hexdigest()
        if h in seen_hashes:
            doc["is_duplicate"] = True
            dropped.append(doc)
        else:
            seen_hashes.add(h)
            doc["is_duplicate"] = False
            kept.append(doc)

    return kept, dropped


# ============================================================
# 6️⃣ Language identification using cc_net.split_by_lang.Classifier
# ============================================================
def load_lid_classifier(model_path: Path) -> split_by_lang.Classifier:
    if not model_path.exists():
        # Download fastText official LID model (compatible with cc_net)
        os.system(
            f"wget -q -O {model_path} "
            "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
        )
    return split_by_lang.Classifier(
        model=model_path,
        field="text",
        out_field="language",
        top=1,
        threshold=0.0,
    )


def run_lid(docs: List[Dict[str, Any]], classifier: split_by_lang.Classifier) -> None:
    for doc in tqdm(docs, desc="🌐 LID via cc_net.Classifier"):
        try:
            classifier(doc)  # Classifier mutates the doc dict
        except Exception:
            doc["language"] = "unk"


# ============================================================
# 7️⃣ Perplexity: MultiSentencePiece + DocLM + PerplexityBucket
# ============================================================
def ensure_lm_files() -> None:
    if not SP_MODEL.exists():
        os.system(
            f"wget -q -O {SP_MODEL} "
            "https://dl.fbaipublicfiles.com/cc_net/lm/en.sp.model"
        )
    if not LM_MODEL.exists():
        os.system(
            f"wget -q -O {LM_MODEL} "
            "https://dl.fbaipublicfiles.com/cc_net/lm/en.arpa.bin"
        )


def build_cutoff_csv(docs: List[Dict[str, Any]]) -> None:
    """
    Build percentile-based cutoff.csv from current sample's perplexity distribution.
    Note: Unlike official global CCNet cutoff, but compatible with PerplexityBucket logic.
    """
    ppl_values = [d.get("perplexity", -1.0) for d in docs if d.get("perplexity", -1.0) > 0]
    if len(ppl_values) > 10:
        percentiles = np.percentile(ppl_values, np.arange(101))
        np.savetxt(
            CUTOFF_CSV,
            percentiles.reshape(-1, 1),
            fmt="%.6f",
            delimiter=",",
            header="en",
            comments="",
        )
        print(f"✅ Created percentile-based cutoff.csv at {CUTOFF_CSV}")
    else:
        # Fallback
        np.savetxt(
            CUTOFF_CSV,
            np.linspace(10, 100, 101).reshape(-1, 1),
            fmt="%.6f",
            delimiter=",",
            header="en",
            comments="",
        )
        print(f"⚠️ Too few PPL samples, wrote fallback cutoff.csv at {CUTOFF_CSV}")


def run_perplexity_and_bucket(docs: List[Dict[str, Any]]) -> None:
    ensure_lm_files()

    sp = perplexity.MultiSentencePiece(
        {"en": SP_MODEL},
        field="text",
        output_field="tokenized",
        normalize=True,
    )

    lm = perplexity.DocLM(
        {"en": LM_MODEL},
        field="tokenized",
        output_field="perplexity",
        normalize=False,
    )

    # Apply SP + LM to English-language docs
    for doc in tqdm(docs, desc="🧠 Perplexity via cc_net.MultiSentencePiece + DocLM"):
        lang = doc.get("language", "unk")
        if lang != "en":
            doc["perplexity"] = -1.0
            continue

        try:
            sp(doc)
            lm(doc)
        except Exception:
            doc["perplexity"] = -1.0

    # Build cutoff.csv from distribution
    build_cutoff_csv(docs)

    # Bucket assignment via PerplexityBucket
    bucketizer = perplexity.PerplexityBucket(CUTOFF_CSV)
    for doc in docs:
        try:
            bucketizer(doc)
        except Exception:
            doc["bucket"] = "all"

    # Remove tokenized field (similar to cc_net.DropKeys)
    for doc in docs:
        if "tokenized" in doc:
            del doc["tokenized"]


# ============================================================
# 8️⃣ Write outputs + zero-width stats
# ============================================================
def write_outputs(
    kept_docs: List[Dict[str, Any]],
    dropped_docs: List[Dict[str, Any]],
) -> None:
    # Zero-width stats
    count_before = sum(count_zwc(d.get("text", "")) for d in kept_docs + dropped_docs)
    count_after = sum(count_zwc(d.get("text", "")) for d in kept_docs)
    retention = count_after / count_before if count_before else 0.0

    # Write cleaned docs (with perplexity & bucket)
    with gzip.open(CLEANED_FILE, "wt", encoding="utf-8") as out:
        for d in kept_docs:
            out.write(json.dumps(d, ensure_ascii=False) + "\n")

    with gzip.open(PPL_FILE, "wt", encoding="utf-8") as out:
        for d in kept_docs:
            out.write(json.dumps(d, ensure_ascii=False) + "\n")

    with DROPPED_FILE.open("w", encoding="utf-8") as out:
        for d in dropped_docs:
            out.write(json.dumps(d, ensure_ascii=False) + "\n")

    wm_dropped = sum(1 for d in dropped_docs if d.get("is_watermarked"))

    print("\n✅ CCNet-style cleaning done.")
    print(
        f"📊 Total input: {len(kept_docs) + len(dropped_docs)} | "
        f"Kept: {len(kept_docs)} | Dropped: {len(dropped_docs)}"
    )
    print(f"💧 Zero-width before: {count_before}")
    print(f"💧 Zero-width after: {count_after}")
    print(f"📊 Zero-width retention rate: {retention:.2%}")
    print(
        f"🧊 Dropped watermarked docs: {wm_dropped} "
        f"({wm_dropped / len(dropped_docs) * 100 if dropped_docs else 0:.2f}%)"
    )
    print(f"💾 Cleaned output: {CLEANED_FILE}")
    print(f"💾 Dropped list: {DROPPED_FILE}")
    print(f"💾 Output with PPL: {PPL_FILE}")


# ============================================================
# 9️⃣ Watermark 1-to-1 retention stats (all documents)
# ============================================================
def load_invisible_chars(mytext_path: Path) -> List[str]:
    with mytext_path.open("r", encoding="utf-8") as f:
        chars = f.read()
    invisible = [c for c in chars if not c.isprintable() and c != "\n"]
    print(f"💧 Loaded {len(invisible)} invisible characters from {mytext_path}")
    return invisible


def watermark_stats(
    kept_docs: List[Dict[str, Any]],
    raw_wm_docs: List[Dict[str, Any]],
    invisible_chars: List[str],
) -> None:
    """
    Strict 1-to-1 mapping:
    - raw_wm_docs[i].wm_idx = i corresponds to invisible_chars[i]
    - If kept_docs include a watermarked doc with wm_idx, count occurrences of its invisible char
    """

    bucket_stats = {"all": {"total": 0, "wm_docs": 0, "wm_zwc": 0}}

    # Stats for cleaned data
    for doc in kept_docs:
        text = doc.get("text", "") or ""
        if not text:
            continue

        bucket_stats["all"]["total"] += 1

        if doc.get("is_watermarked") and doc.get("wm_idx") is not None:
            wm_idx = doc["wm_idx"]
            if 0 <= wm_idx < len(invisible_chars):
                ch = invisible_chars[wm_idx]
                bucket_stats["all"]["wm_docs"] += 1
                bucket_stats["all"]["wm_zwc"] += text.count(ch)

    # Stats for raw data
    raw_total_wm_docs = len(raw_wm_docs)
    raw_total_zwc = 0
    for raw in raw_wm_docs:
        wm_idx = raw["wm_idx"]
        if 0 <= wm_idx < len(invisible_chars):
            ch = invisible_chars[wm_idx]
            raw_total_zwc += raw.get("watermarked", "").count(ch)

    print("\n📊 Watermark Retention (all documents)")
    print(f"Total raw watermarked docs: {raw_total_wm_docs}")
    print(f"Total raw invisible chars (1-to-1): {raw_total_zwc}")

    data = bucket_stats["all"]
    wm_docs = data["wm_docs"]
    wm_zwc = data["wm_zwc"]
    doc_ratio = wm_docs / data["total"] * 100 if data["total"] else 0.0
    zwc_retention = wm_zwc / raw_total_zwc * 100 if raw_total_zwc else 0.0

    print(f"\n—— ALL ——")
    print(f"📄 Total docs: {data['total']}")
    print(f"💧 Watermarked docs: {wm_docs} ({doc_ratio:.2f}%)")
    print(f"🔢 Invisible chars retained: {wm_zwc}")
    print(f"📈 Retention vs. raw: {zwc_retention:.2f}%")


# ============================================================
# 🔚 main
# ============================================================
def main():
    # 1. Load original docs & raw watermark list
    docs, raw_wm_docs = load_docs(INPUT_FILE)

    # 2. Zero-width stats before cleaning
    count_before = sum(count_zwc(d["text"]) for d in docs)
    print(f"💧 Zero-width before cleaning: {count_before}")

    # 3. Document-level dedup via cc_net
    kept_docs, dropped_docs = ccnet_doc_dedup(docs)

    # 4. LID via cc_net Classifier
    lid_classifier = load_lid_classifier(LID_MODEL_PATH)
    run_lid(kept_docs, lid_classifier)

    # 5. Perplexity + bucket via cc_net
    run_perplexity_and_bucket(kept_docs)

    # 6. Write outputs + zero-width stats
    write_outputs(kept_docs, dropped_docs)

    # 7. Watermark retention stats
    if MYTEXT_PATH.exists():
        invisible_chars = load_invisible_chars(MYTEXT_PATH)
        watermark_stats(kept_docs, raw_wm_docs, invisible_chars)
    else:
        print(f"⚠️ {MYTEXT_PATH} not found, skip watermark stats.")


if __name__ == "__main__":
    main()
